In [ ]:
!pip install -U langchain
!pip install langchain-classic


In [1]:
print("--- เริ่มรันโปรแกรมแล้ว! ---")

--- เริ่มรันโปรแกรมแล้ว! ---


In [2]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

print("1. กำลังโหลดเอกสาร PDF...")
loader = PyMuPDFLoader("Guidlines_Application.pdf")
documents = loader.load()

print("2. กำลังหั่นเอกสาร (Chunking)...")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=2500, chunk_overlap=400)
chunks = text_splitter.split_documents(documents)

print("3. กำลังแปลงข้อความเป็น Vector และเซฟลงเครื่อง...")
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

# สร้างและเซฟลงโฟลเดอร์ chroma_db
vector_db = Chroma.from_documents(chunks, embeddings, persist_directory="./chroma_db")

print("✅ สร้างฐานข้อมูลเสร็จเรียบร้อย! ไปรัน Cell ที่ 2 เพื่อถามคำถามได้เลย")

d:\Code\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1. กำลังโหลดเอกสาร PDF...
2. กำลังหั่นเอกสาร (Chunking)...
3. กำลังแปลงข้อความเป็น Vector และเซฟลงเครื่อง...


C:\Users\Deler\AppData\Local\Temp\ipykernel_8116\3784172242.py:15: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 15710.36it/s]


✅ สร้างฐานข้อมูลเสร็จเรียบร้อย! ไปรัน Cell ที่ 2 เพื่อถามคำถามได้เลย


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.llms import Ollama
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA

print("1. โหลด Embedding และฐานข้อมูลเดิมขึ้นมา (ไม่ต้องอ่าน PDF ใหม่แล้ว)...")
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

# โหลดฐานข้อมูลจากโฟลเดอร์ ./chroma_db ขึ้นมาใช้เลย
vector_db = Chroma(persist_directory="./chroma_db", embedding_function=embeddings)

print("2. กำลังเชื่อมต่อ Ollama (Qwen 2.5)...")
llm = Ollama(model="qwen2.5")

# 3. ตั้งค่า Prompt บังคับภาษาไทย
template = """คุณคือผู้เชี่ยวชาญด้านให้คำปรึกษาทุนการศึกษา
จงอ่านข้อมูล (Context) ด้านล่างนี้ ซึ่งอาจเป็นภาษาอังกฤษหรือภาษาญี่ปุ่น แล้วนำมาอธิบายโดยละเอียดเพื่อตอบคำถามของผู้ใช้ โดยมีข้อกำหนดในการตอบเพิ่มเติมดังนี้
1.ต้องตอบกลับเป็น "ภาษาไทย" สละสลวยเท่านั้น! (ห้ามพิมพ์ภาษาจีน หรือ ญี่ปุ่น ปะปนมาเด็ดขาด)
2.หากไม่สามารถหาคำตอบได้จากการอ่านเอกสาร ห้ามคาดเดาหรือสร้างข้อมูลขึ้นมาเองโดยเด็ดขาด ให้ตอบว่า "ไม่ได้ระบุไว้ในเอกสาร"
3.หากพูดถึงจำนวนเงินทุนการศึกษา หรือทุนค่าใช้จ่ายอื่น หากระบุเป็นหน่วย เยน (JPY) ให้ตอบกลับมาเป็นหน่วยเงิน เยน (JPY) เท่านั้นห้ามตอบเป็นหน่วยอื่นเด็ดขาด

ข้อมูลอ้างอิง (Context):
---------------------
{context}
---------------------

คำถาม: {question}

คำสั่งสำคัญ: 
- จงสรุปคำตอบเป็น "ภาษาไทยเท่านั้น" โดยอธิบายโดยละเอียดอย่างเช่น
- หากพบว่าตนเองกำลังจะพิมพ์ภาษาจีน (中文) หรือเกาหลี (한국어) ให้หยุดทันทีและแปลเป็นไทย
- ห้ามมีตัวอักษรจีนหรือเกาหลีปรากฏในคำตอบเด็ดขาด!

คำตอบ (ภาษาไทย):"""

QA_PROMPT = PromptTemplate.from_template(template)

# 4. สร้างระบบถาม-ตอบ
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vector_db.as_retriever(search_kwargs={"k": 4}), 
    return_source_documents=True,
    chain_type_kwargs={"prompt": QA_PROMPT}
)

# ==========================================
# 🎯 เปลี่ยนคำถามตรงนี้ แล้วกดรัน Cell นี้ได้รัวๆ เลยครับ!
query = "ช่วยอธิบายทุน MEXT โดยละเอียดให้ที"
# ==========================================

print(f"\nคำถาม: {query}")
print("กำลังคิดคำตอบ (อาจใช้เวลาสักครู่)...")

# สั่งรันหาคำตอบ
result = qa_chain.invoke({"query": query})

print("\n=== คำตอบ ===")
print(result["result"])

1. โหลด Embedding และฐานข้อมูลเดิมขึ้นมา (ไม่ต้องอ่าน PDF ใหม่แล้ว)...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 17652.53it/s]
C:\Users\Deler\AppData\Local\Temp\ipykernel_7220\3599400265.py:11: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_db = Chroma(persist_directory="./chroma_db", embedding_function=embeddings)
C:\Users\Deler\AppData\Local\Temp\ipykernel_7220\3599400265.py:14: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="qwen2.5")


2. กำลังเชื่อมต่อ Ollama (Qwen 2.5)...

คำถาม: ช่วยอธิบายทุน MEXT โดยละเอียดให้ที
กำลังคิดคำตอบ (อาจใช้เวลาสักครู่)...

=== คำตอบ ===
ทุน MEXT เป็นทุนการศึกษาสำหรับนักศึกษานานาชาติที่เข้าสมัครและผ่านขั้นตอนการคัดเลือก ซึ่งมีจุดประสงค์เพื่อสนับสนุนนักศึกษาให้สามารถศึกษาในประเทศญี่ปุ่น ในระหว่างที่ได้รับทุนนี้ ผู้สมัครจำเป็นต้องปฏิบัติตามข้อกำหนดและระเบียบที่ระบุไว้ในการปฏิญาณ รวมถึงการมีส่วนร่วมในกิจกรรมที่โรงเรียนและชุมชนเพื่อเสริมสร้างความเข้าใจระหว่างประเทศญี่ปุ่นและประเทศของตนเอง ตลอดจนพยายามรักษาความสัมพันธ์ด้านการศึกษากับมหาวิทยาลัยที่ผู้สมัครเข้าศึกษา

เมื่อผ่านขั้นตอนการคัดเลือกในครั้งแรก (First Screening) ทางสำนักงานญี่ปุ่นประจำประเทศของผู้สมัครจะทำการประเมินเพื่อคัดเลือกผู้สมัครให้เป็นผู้รับทุน MEXT ในขั้นตอนการคัดเลือกในครั้งที่สอง (Second Screening) สำนักงานต่างประเทศของญี่ปุ่นจะทำการประเมินผลตามที่ได้รับจากครั้งแรกและตกลงกับมหาวิทยาลัยที่ผู้สมัครระบุไว้ในแบบฟอร์มคำขอเพื่อเลือกมหาวิทยาลัยสำหรับผู้สมัคร แต่หากไม่มีการตอบรับจากมหาวิทยาลัยที่ผู้สมัครระบุไว้ อาจไม่สามารถให้การศึ

In [11]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.llms import Ollama
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA

print("1. โหลด Embedding และฐานข้อมูลเดิมขึ้นมา (ไม่ต้องอ่าน PDF ใหม่แล้ว)...")
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

# โหลดฐานข้อมูลจากโฟลเดอร์ ./chroma_db ขึ้นมาใช้เลย
vector_db = Chroma(persist_directory="./chroma_db", embedding_function=embeddings)

print("2. กำลังเชื่อมต่อ Ollama (Qwen 2.5)...")
llm = Ollama(model="qwen2.5")

# 3. ตั้งค่า Prompt บังคับภาษาไทย
template = """คุณคือผู้เชี่ยวชาญด้านให้คำปรึกษาทุนการศึกษา
จงอ่านข้อมูล (Context) ด้านล่างนี้ ซึ่งอาจเป็นภาษาอังกฤษหรือภาษาญี่ปุ่น แล้วนำมาอธิบายโดยละเอียดเพื่อตอบคำถามของผู้ใช้ โดยมีข้อกำหนดในการตอบเพิ่มเติมดังนี้
1.หากไม่สามารถหาคำตอบได้จากการอ่านเอกสาร ห้ามคาดเดาหรือสร้างข้อมูลขึ้นมาเองโดยเด็ดขาด ให้ตอบว่า "ไม่ได้ระบุไว้ในเอกสาร"
2.ต้องตอบกลับเป็น "ภาษาไทย" สละสลวยเท่านั้น! (ห้ามพิมพ์ภาษาจีน หรือ ญี่ปุ่น ปะปนมาเด็ดขาด)
3.หากพูดถึงจำนวนเงินทุนการศึกษา หรือทุนค่าใช้จ่ายอื่น หากระบุเป็นหน่วย เยน (JPY) ให้ตอบกลับมาเป็นหน่วยเงิน เยน (JPY) เท่านั้นห้ามตอบเป็นหน่วยอื่นเด็ดขาด
4.สำคัญมาก: จงวิเคราะห์ "สถานะของผู้ถาม" (เช่น กำลังเรียน, ทำงานอยู่) แล้วตอบให้ตรงประเด็นที่สุดว่าสถานะแบบนี้สมัครได้หรือไม่ เพราะเหตุใด (อ้างอิงคุณสมบัติด้านวุฒิการศึกษาเป็นหลัก) อย่านำข้อห้ามที่ไม่เกี่ยวข้องมาตอบถ้าไม่จำเป็น


ข้อมูลอ้างอิง (Context):
---------------------
{context}
---------------------

คำถาม: {question}

คำสั่งสำคัญ: 
- จงอธิบายคำตอบเป็น "ภาษาไทยเท่านั้น" โดยอธิบายโดยละเอียด
- หากพบว่าตนเองกำลังจะพิมพ์ภาษาจีน (中文) หรือเกาหลี (한국어) ให้หยุดทันทีและแปลเป็นไทย
- ห้ามมีตัวอักษรจีนหรือเกาหลีปรากฏในคำตอบเด็ดขาด!
- ให้อ้างอิงในการตอบด้วยว่าจากเอกสารอะไร หน้าไหน ในหัวข้ออะไร
- จงวิเคราะห์ "สถานะของผู้ถาม" (เช่น กำลังเรียน, ทำงานอยู่) แล้วตอบให้ตรงประเด็นที่สุดว่าสถานะแบบนี้สมัครได้หรือไม่ เพราะเหตุใด (อ้างอิงคุณสมบัติด้านวุฒิการศึกษาเป็นหลัก) อย่านำข้อห้ามที่ไม่เกี่ยวข้องมาตอบถ้าไม่จำเป็น

คำตอบ (ภาษาไทย):"""

QA_PROMPT = PromptTemplate.from_template(template)

# 4. สร้างระบบถาม-ตอบ
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vector_db.as_retriever(search_kwargs={"k": 6}), 
    return_source_documents=True,
    chain_type_kwargs={"prompt": QA_PROMPT}
)


query = "ถ้าฉันเป็นนักศึกษาที่กำลังจะจบปริญญาตรี (กำลังศึกษาในระดับปริญญาตรีชั้นปีที่ 4 และคาดว่าจะสำเร็จการศึกษาในช่วงก่อนที่จะรับทุนและไปศึกษาต่อ) สามารถสมัครทุน MEXT นี้ได้ไหม"

print(f"\nคำถาม: {query}")
print("กำลังคิดคำตอบ (อาจใช้เวลาสักครู่)...")

result = qa_chain.invoke({"query": query})

print("\n=== คำตอบ ===")
print(result["result"])

1. โหลด Embedding และฐานข้อมูลเดิมขึ้นมา (ไม่ต้องอ่าน PDF ใหม่แล้ว)...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 11285.10it/s]


2. กำลังเชื่อมต่อ Ollama (Qwen 2.5)...

คำถาม: ถ้าฉันเป็นนักศึกษาที่กำลังจะจบปริญญาตรี (กำลังศึกษาในระดับปริญญาตรีชั้นปีที่ 4 และคาดว่าจะสำเร็จการศึกษาในช่วงก่อนที่จะรับทุนและไปศึกษาต่อ) สามารถสมัครทุน MEXT นี้ได้ไหม
กำลังคิดคำตอบ (อาจใช้เวลาสักครู่)...

=== คำตอบ ===
ตามข้อมูลในเอกสาร MEXT ระบุไว้ในหัวข้อยอดผู้สมัครที่ไม่เข้าคุณสมบัติ (8. Non-eligibility) มีข้อ ⑦ ที่ระบุว่า "Those who are expected to graduate at the time of application and cannot satisfy the condition for academic background by the deadline given" แปลว่า ผู้สมัครที่คาดว่าจะสำเร็จการศึกษาในช่วงเวลาของคำขอทุนและไม่สามารถที่จะตรงตามคุณสมบัติทางวุฒิการศึกษาโดยมีกำหนดระยะเวลาที่ให้มา ไม่มีสิทธิ์ในการสมัครทุน MEXT.

เนื่องจากคุณกำลังศึกษาในระดับปริญญาตรีชั้นปีที่ 4 และคาดว่าจะสำเร็จการศึกษาในช่วงก่อนรับทุนและไปศึกษาต่อ ซึ่งตรงตามสถานะที่ระบุไว้ในข้อ ⑦ ของเอกสาร MEXT คุณไม่มีสิทธิ์ในการสมัครทุน MEXT. กรุณาตรวจสอบว่ามีการวางแผนสำเร็จการศึกษาล่วงหน้าหรือไม่ และให้มั่นใจว่าคุณจะสามารถตรงตามเงื่อนไขทางวุฒิการศึกษาโดยมีกำหนดระยะเ

In [17]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.llms import Ollama
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA

print("1. โหลด Embedding และฐานข้อมูลเดิมขึ้นมา (ไม่ต้องอ่าน PDF ใหม่แล้ว)...")
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

# โหลดฐานข้อมูลจากโฟลเดอร์ ./chroma_db ขึ้นมาใช้เลย
vector_db = Chroma(persist_directory="./chroma_db", embedding_function=embeddings)

print("2. กำลังเชื่อมต่อ Ollama (Qwen 2.5)...")
llm = Ollama(model="qwen2.5")

# 3. ตั้งค่า Prompt บังคับภาษาไทย
template = """คุณคือผู้เชี่ยวชาญด้านให้คำปรึกษาทุนการศึกษา
จงอ่านข้อมูล (Context) ด้านล่างนี้ ซึ่งอาจเป็นภาษาอังกฤษหรือภาษาญี่ปุ่น แล้วนำมาอธิบายโดยละเอียดเพื่อตอบคำถามของผู้ใช้ โดยมีข้อกำหนดในการตอบเพิ่มเติมดังนี้
1.หากไม่สามารถหาคำตอบได้จากการอ่านเอกสาร ห้ามคาดเดาหรือสร้างข้อมูลขึ้นมาเองโดยเด็ดขาด ให้ตอบว่า "ไม่ได้ระบุไว้ในเอกสาร"
2.ต้องตอบกลับเป็น "ภาษาไทย" สละสลวยเท่านั้น! (ห้ามพิมพ์ภาษาจีน หรือ ญี่ปุ่น ปะปนมาเด็ดขาด)
3.หากพูดถึงจำนวนเงินทุนการศึกษา หรือทุนค่าใช้จ่ายอื่น หากระบุเป็นหน่วย เยน (JPY) ให้ตอบกลับมาเป็นหน่วยเงิน เยน (JPY) เท่านั้นห้ามตอบเป็นหน่วยอื่นเด็ดขาด
4.สำคัญมาก: จงวิเคราะห์ "สถานะของผู้ถาม" (เช่น กำลังเรียน, ทำงานอยู่) แล้วตอบให้ตรงประเด็นที่สุดว่าสถานะแบบนี้สมัครได้หรือไม่ เพราะเหตุใด (อ้างอิงคุณสมบัติด้านวุฒิการศึกษาเป็นหลัก) อย่านำข้อห้ามที่ไม่เกี่ยวข้องมาตอบถ้าไม่จำเป็น


ข้อมูลอ้างอิง (Context):
---------------------
{context}
---------------------

คำถาม: {question}

คำสั่งสำคัญ: 
- จงอธิบายคำตอบเป็น "ภาษาไทยเท่านั้น" โดยอธิบายโดยละเอียด
- หากพบว่าตนเองกำลังจะพิมพ์ภาษาจีน (中文) หรือเกาหลี (한국어) ให้หยุดทันทีและแปลเป็นไทย
- ห้ามมีตัวอักษรจีนหรือเกาหลีปรากฏในคำตอบเด็ดขาด!
- ให้อ้างอิงในการตอบด้วยว่าจากเอกสารอะไร หน้าไหน ในหัวข้ออะไร
- จงวิเคราะห์ "สถานะของผู้ถาม" (เช่น กำลังเรียน, ทำงานอยู่) แล้วตอบให้ตรงประเด็นที่สุดว่าสถานะแบบนี้สมัครได้หรือไม่ เพราะเหตุใด (อ้างอิงคุณสมบัติด้านวุฒิการศึกษาเป็นหลัก) อย่านำข้อห้ามที่ไม่เกี่ยวข้องมาตอบถ้าไม่จำเป็น

คำตอบ (ภาษาไทย):"""

QA_PROMPT = PromptTemplate.from_template(template)

# 4. สร้างระบบถาม-ตอบ
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vector_db.as_retriever(search_kwargs={"k": 6}), 
    return_source_documents=True,
    chain_type_kwargs={"prompt": QA_PROMPT}
)


query = "ในการสมัครต้องมีการเตรียมเอกสารอะไรบ้าง จำนวนเท่าไร อย่างไร"

print(f"\nคำถาม: {query}")
print("กำลังคิดคำตอบ (อาจใช้เวลาสักครู่)...")

result = qa_chain.invoke({"query": query})

print("\n=== คำตอบ ===")
print(result["result"])

1. โหลด Embedding และฐานข้อมูลเดิมขึ้นมา (ไม่ต้องอ่าน PDF ใหม่แล้ว)...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 22050.65it/s]


2. กำลังเชื่อมต่อ Ollama (Qwen 2.5)...

คำถาม: ในการสมัครต้องมีการเตรียมเอกสารอะไรบ้าง จำนวนเท่าไร อย่างไร
กำลังคิดคำตอบ (อาจใช้เวลาสักครู่)...

=== คำตอบ ===
ในการสมัครได้รับทุนการศึกษาจากญี่ปุ่น ผู้สมัครต้องเตรียมและส่งเอกสารตามรายละเอียดในเอกสาร โดยมีเอกสารหลักๆ ดังนี้:

1. เอกสารที่ต้องเตรียม (ไม่ว่าสถานะของผู้สมัครเป็นอย่างไร):

- แบบฟอร์มการสมัคร (Application Form) : 1 ฉบับ พร้อมสำเนา 2 ชุด
- แบบฟอร์มpreferences ของการเข้าเรียน (Placement Preference Application Form): 1 ฉบับ (เฉพาะผู้ที่ถูกคัดเลือกจากครั้งแรก)
- แบบฟอร์มการศึกษาและแผนวิจัย (Field of Study and Research Plan) : 1 ฉบับ พร้อมสำเนา 2 ชุด
- ประวัติการเรียนของมหาวิทยาลัยที่ผ่านมา (academic transcript): 1 ฉบับ พร้อมสำเนา 2 ชุด 
- ใบรับปริญญาหรือหนังสือรับรองปริญญา (certificate of graduation or degree certificate): 1 ฉบับ พร้อมสำเนา 2 ชุด
- จดหมายเชิงว่าทั้งหมด (recommendation letter) : 1 ฉบับ พร้อมสำเนา 2 ชุด (เฉพาะผู้สมัครที่กำลังทำงาน)
- ภาพถ่ายขนาด 4.5 x 3.5 ซม. หน้าอกเต็ม (photograph): 1 ฉบับ
- ใบรับรองการตรวจสุขภาพ 